## Derivative-Constrained PINN — Gibson–Schwartz convenience-yield inversion

Ported from a colleague's AC-PINN calibration notebook, originally built for SABR / Dupire-local-vol option calibration. Kept as-is because it's problem-agnostic: the network architecture, the optax optimizer, and the WAMOL loss-balancing machinery (self-adaptive per-point weights + gradient-norm loss balancing). Stripped entirely: every Black-Scholes / SABR / Dupire-local-vol function (`bs`, `black`, `d1_`, `d2_`, `lv_sqr`, `SabrVolHagan`, `bs_iv`, the arbitrage-heatmap and vol-surface plots, `get_data`'s SABR sampler) — none of it applies to Gibson-Schwartz.

**One exception to "everything ported into this notebook":** the closed-form GS coefficients (`GSParams`, `B_coeff`, `A_coeff`) are imported from `gs_wamol.physics.gibson_schwartz` rather than re-typed here. Per `CLAUDE.md` sec. 3, those coefficients are exact and verified against Schwartz (1997) eqs. (18)–(20) — retyping them risks a silent transcription bug, and `monte_carlo.ipynb` already depends on that exact module.

**Loss terms now come from `docs/mathematical_derivations-5.pdf`, not from my own guess.** An earlier version of this notebook improvised an "OU drift-consistency" physics term because the network's exact I/O contract hadn't been pinned down yet — that guess turned out to be wrong on two counts (it collapsed `delta_hat` to a flat line at `alpha_P` in testing, and it wasn't actually what you'd derived). The PDF fixes both:
- **Network I/O**: `(t, S) -> delta_hat` — two inputs, not one. Eq. (32) writes the recovered yield explicitly as `delta_hat(t_i, S_i)`.
- **`e_acc` (their `L_t`, eq. 32)**: MSE of `ln S + B(tau) delta_hat(t,S) + A(tau)` against `ln F_obs`, in log-price space.
- **`e_pde` (their `L_f`, eq. 33/35)**: a genuine PDE residual, obtained by automatically differentiating the *composite* `F_hat(t,S,tau) := S * exp(B(tau) delta_hat(t,S) + A(tau))` through the network -- not the abstract closed-form's own `(S, delta, tau)` arguments (which would make the residual identically zero, since it IS the PDE's exact solution by construction). This is the same trick the original notebook's `call_derivatives` used: differentiate `bs(..., vol=network(x))` through the network, not the raw Black-Scholes formula.
- **Inequality terms (their `L_h`, Section 6)**: three concrete constraints -- cash-and-carry upper bound (eq. 22), reverse-cash-and-carry lower bound (eq. 25), and a convenience-yield floor (eq. 28). Implemented below, with one deliberate deviation from the PDF flagged inline (their `delta_min = -0.3` is specific to their own simulated draw and leaks `delta_true`-adjacent information if copied blindly -- see the Loss terms cell).
- **`L_b`**: the PDF's own footnote says "Definition to be supplied." Left unimplemented here too -- not something I'm inventing on your behalf.

In [29]:
import os
import pickle
import time
import json
from typing import Callable, Dict, Optional, Tuple, Union

import jax
import jax.numpy as jnp
from jax import grad, jacrev, jit, lax, random, vmap
from jax.nn.initializers import glorot_normal, normal, zeros
from jax.flatten_util import ravel_pytree
from jax.tree_util import tree_leaves, tree_map
import jax.example_libraries.optimizers as jax_optimizers

import optax
import ml_collections
from flax import linen as nn
from flax.training import train_state, orbax_utils
from flax.serialization import to_state_dict, from_state_dict
import orbax.checkpoint

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from gs_wamol.physics.gibson_schwartz import GSParams, B_coeff, A_coeff

np.set_printoptions(precision=4, suppress=True)
print("JAX:", jax.__version__, "| x64 enabled:", jax.config.jax_enable_x64,
      "| default dtype:", jnp.ones(1).dtype)

JAX: 0.9.1 | x64 enabled: False | default dtype: float32


### Network architecture

Core code inherited from https://github.com/khoshisashi/AC-PINNs/blob/main/AC_PINNs_IVS_calibration.ipynb for the original SABR / Dupire-local-vol problem by Khoshisashi et al. 2023. in his ACPINN repo.

`x = nn.softplus(x)` was used in the orginal to force a positive output. Convenience yield has no such sign constraint, so that line is removed in the modified network below.

In [30]:
activation_fn = {
    "tanh": jnp.tanh,
    "sin": jnp.sin,
}


def _get_activation(name):
    if name in activation_fn:
        return activation_fn[name]
    raise NotImplementedError(f"Activation {name} not supported yet!")


def _weight_fact(init_fn, mean, stddev):
    def init(key, shape):
        key1, key2 = random.split(key)
        w = init_fn(key1, shape)
        g = mean + normal(stddev)(key2, (shape[-1],))
        g = jnp.exp(g)
        v = w / g
        return g, v
    return init


class Dense(nn.Module):
    features: int
    kernel_init: Callable = glorot_normal()
    bias_init: Callable = zeros
    reparam: Union[None, Dict] = None

    @nn.compact
    def __call__(self, x):
        if self.reparam is None:
            kernel = self.param(
                "kernel", self.kernel_init, (x.shape[-1], self.features)
            )
        elif self.reparam["type"] == "weight_fact":
            g, v = self.param(
                "kernel",
                _weight_fact(
                    self.kernel_init,
                    mean=self.reparam["mean"],
                    stddev=self.reparam["stddev"],
                ),
                (x.shape[-1], self.features),
            )
            kernel = g * v
        bias = self.param("bias", self.bias_init, (self.features,))
        y = jnp.dot(x, kernel) + bias
        return y


class MLP(nn.Module):
    arch_name: Optional[str] = "MLP"
    hidden_dim: Tuple[int] = (32, 16)
    out_dim: int = 1
    activation: str = "tanh"
    periodicity: Union[None, Dict] = None
    fourier_emb: Union[None, Dict] = None
    reparam: Union[None, Dict] = None

    def setup(self):
        self.activation_fn = _get_activation(self.activation)

    @nn.compact
    def __call__(self, x):
        for i in range(len(self.hidden_dim)):
            x = Dense(features=self.hidden_dim[i], reparam=self.reparam)(x)
            x = self.activation_fn(x)
        x = Dense(features=self.out_dim, reparam=self.reparam)(x)
        # original: x = nn.softplus(x) -- removed, delta_hat is unconstrained in sign
        return x


class ModifiedMLP(nn.Module):
    arch_name: Optional[str] = "ModifiedMLP"
    hidden_dim: Tuple[int] = (32, 16)
    out_dim: int = 1
    activation: str = "tanh"
    periodicity: Union[None, Dict] = None
    fourier_emb: Union[None, Dict] = None
    reparam: Union[None, Dict] = None

    def setup(self):
        self.activation_fn = _get_activation(self.activation)

    @nn.compact
    def __call__(self, x):
        u = Dense(features=self.hidden_dim[0], reparam=self.reparam)(x)
        v = Dense(features=self.hidden_dim[0], reparam=self.reparam)(x)

        u = self.activation_fn(u)
        v = self.activation_fn(v)

        for i in range(len(self.hidden_dim)):
            x = Dense(features=self.hidden_dim[i], reparam=self.reparam)(x)
            x = self.activation_fn(x)
            x = x * u + (1 - x) * v

        x = Dense(features=self.out_dim, reparam=self.reparam)(x)
        # original: x = nn.softplus(x) -- removed, delta_hat is unconstrained in sign
        return x


def ann_gen(config):
    ann = None
    reparam = None
    if config.ann_reparam == "weight_fact":
        reparam = ml_collections.ConfigDict({"type": "weight_fact", "mean": 0.5, "stddev": 0.1})

    if config.ann_str == "MLP":
        ann = MLP(arch_name=config.ann_str,
                  hidden_dim=config.ann_hidden_dim,
                  out_dim=config.ann_out_dim,
                  activation=config.ann_activation_str,
                  periodicity=config.ann_periodicity,
                  fourier_emb=config.ann_fourier_emb,
                  reparam=reparam)
    elif config.ann_str == "ModifiedMLP":
        ann = ModifiedMLP(arch_name=config.ann_str,
                  hidden_dim=config.ann_hidden_dim,
                  out_dim=config.ann_out_dim,
                  activation=config.ann_activation_str,
                  periodicity=config.ann_periodicity,
                  fourier_emb=config.ann_fourier_emb,
                  reparam=reparam)
    return ann

In [31]:
# --- Two networks (S6, eq. 30) ---------------------------------------------
#   F_hat_theta : (S, delta, tau) -> R+    the pricing SURFACE  (in_dim 3)
#   delta_hat_phi : t -> R                 the latent PATH      (in_dim 1)
#
# Why two, and why the closed form is gone from the objective (S6 remark 2):
# the exponential-affine solution solves the pricing PDE identically for EVERY
# value of the coordinate delta, so composing it with delta_hat_phi would drive
# L_f and L_b to zero by construction and reduce the method to penalised least
# squares. Verified numerically: the self-consistent residual of the closed form
# is 4.7e-13 (machine eps at x64). The PDE only has content when it acts on a
# free surface that is ABLE to violate it.
#
# delta_hat_phi takes t ALONE (S6 remark 1). Admitting S would "let the network
# launder price information into the yield estimate and blur its interpretation
# as a filtered state"; the S-delta dependence is already carried by rho in the
# dynamics that L_f enforces.


def build_nets(config):
    """Return (surface_ann, path_ann). Different input dims, so no single
    `ann_in_dim` any more."""
    reparam = None
    if config.ann_reparam == "weight_fact":
        reparam = ml_collections.ConfigDict({"type": "weight_fact", "mean": 0.5, "stddev": 0.1})

    surface_ann = MLP(arch_name="MLP",
                      hidden_dim=config.surface_hidden_dim,
                      out_dim=1,
                      activation=config.ann_activation_str,
                      reparam=reparam)
    path_ann = MLP(arch_name="MLP",
                   hidden_dim=config.path_hidden_dim,
                   out_dim=1,
                   activation=config.ann_activation_str,
                   reparam=reparam)
    return surface_ann, path_ann


def make_fns(surface_ann, path_ann, params):
    """Bind params into the three callables the objective needs.

    Hard-enforced initial condition (S11 eq. 47):

        F_hat_theta(S, delta, tau) = S * exp(tau * N_theta(S, delta, tau))

    At tau = 0 this gives F_hat = S identically, so L_b vanishes by construction
    and is DROPPED from the objective (41) -- there is no e_b component below.
    It also gives positivity (F_hat > 0) for free, and makes the log-price used
    by L_t linear in the raw output:  log F_hat = log S + tau * N_theta.
    """
    def N_theta(S, delta, tau):
        return surface_ann.apply(params["surface"], normalize_surface(S, delta, tau))[0]

    def log_F_hat_fn(S, delta, tau):
        return jnp.log(S) + tau * N_theta(S, delta, tau)      # eq. 47 in log form

    def F_hat_fn(S, delta, tau):
        return jnp.exp(log_F_hat_fn(S, delta, tau))

    def delta_hat_fn(t):
        return path_ann.apply(params["path"], normalize_time(t))[0]

    return F_hat_fn, delta_hat_fn, log_F_hat_fn


def init_nets(config, key):
    """Joint parameter pytree -- one TrainState covers both nets, so a single
    apply_gradients step updates theta and phi together (trained jointly, S6)."""
    surface_ann, path_ann = build_nets(config)
    k_s, k_p = random.split(key, 2)
    params = {
        "surface": surface_ann.init(k_s, jnp.ones((3,))),
        "path": path_ann.init(k_p, jnp.ones((1,))),
    }
    return surface_ann, path_ann, params


### Optimiser

Core code inherited from https://github.com/khoshisashi/AC-PINNs/blob/main/AC_PINNs_IVS_calibration.ipynb for the original SABR / Dupire-local-vol problem by Khoshisashi et al. 2023. in his ACPINN repo.

In [32]:
optimizer = "Adam"
beta1 = 0.9
beta2 = 0.999
adam_eps = 1e-8
learning_rate = 1e-3
decay_rate = 0.9
decay_steps = 2000

lr = optax.exponential_decay(
        init_value=learning_rate,
        transition_steps=decay_steps,
        decay_rate=decay_rate,
    )
tx = optax.adam(
    learning_rate=lr, b1=beta1, b2=beta2, eps=adam_eps
)

### Input Data

Monte Carlo simulated data generated by `monte_carlo.ipynb` and saved to `data/gs_mc_data.npz`.

Input data is a 2D array of `(t, S)` pairs, and the corresponding observed forward prices `F_obs` at those points. The network will learn to predict the convenience yield `delta_hat(t, S)` that best fits the observed forward prices under the Gibson-Schwartz model.

In [33]:
def get_data(path_id=0, path="../data/input/synthetic/mc_data.pkl"):

    with open(path, "rb") as f:
        mc_data = pickle.load(f)

    taus = mc_data["taus"]                             
    t_train = mc_data["t_grid"]                          
    S_train = mc_data["S"][path_id]                      
    log_F_obs_train = mc_data["log_F_obs"][path_id]

    # EVAL TARGET ONLY -- never fed to the network or error().
    delta_true = mc_data["delta_true"][path_id]           

    p_Q = mc_data["params_Q"]                              
    kappa_P = mc_data["params_P"]["kappa"]                 
    alpha_P = mc_data["params_P"]["alpha_P"]               
    sigma2_P = mc_data["params_P"]["sigma2"]               

    return t_train, S_train, taus, log_F_obs_train, delta_true, p_Q, kappa_P, alpha_P, sigma2_P



In [34]:
# --- Bind the panel and the globals the objective closes over --------------
(t_train, S_train, taus, log_F_obs_train,
 delta_true, p_Q, kappa_P, alpha_P, sigma2_P) = get_data(path_id=0)

# delta_true is deliberately NOT in `data`: it is the evaluation target and must
# never reach a loss term (CLAUDE.md S6.3 -- the true delta path is sacred).
data = (t_train, S_train, log_F_obs_train)

n_dates, n_taus = log_F_obs_train.shape


# --- Input normalisation ---------------------------------------------------
# Raw inputs span t in [0, 10], S in [36, 165], tau in [0.08, 2]. tanh saturates
# outside roughly [-3, 3]: measured on the old single net, 96% of first-layer
# units were saturated (mean tanh' = 0.008) and delta_hat was constant to 5e-3
# across the whole panel. Every network input is rescaled to about unit spread.
#
#   S   -> (log S - mean)/std   log first: S is lognormal under GS, so log S is
#                               the naturally symmetric coordinate
#   delta -> delta / DELTA_SCALE
#   tau -> tau / TAU_SCALE
#   t   -> t / T_SCALE
#
# Stats come from the training panel. No held-out split exists here (one path,
# full panel) so this is not leakage -- freeze them from train only if you later
# evaluate on a second path.
T_SCALE = float(t_train[-1])
LOG_S_MEAN = float(jnp.mean(jnp.log(S_train)))
LOG_S_STD = float(jnp.std(jnp.log(S_train)))
DELTA_SCALE = 0.5
TAU_SCALE = float(taus.max())


def normalize_surface(S, delta, tau):
    """Physical (S, delta, tau) -> surface-network input coordinates.

    Applied at the network boundary, never to the stored arrays: rPDE (eq. 44)
    differentiates F_hat_theta w.r.t. PHYSICAL S and delta, and eq. 47 needs
    physical S for the S*exp(tau*N) prefactor. Autodiff chains through this map
    on its own, so the residual stays correct.
    """
    return jnp.array([(jnp.log(S) - LOG_S_MEAN) / LOG_S_STD,
                      delta / DELTA_SCALE,
                      tau / TAU_SCALE])


def normalize_time(t):
    """Physical t -> path-network input. Shape (1,): the path net is 1-D (S6)."""
    return jnp.atleast_1d(t / T_SCALE)


# --- Collocation mesh for L_f (S10, eq. 45) --------------------------------
# delta is an INDEPENDENT COORDINATE here, sampled over a range -- NOT
# delta_hat_phi(t). Eq. 44: "The latent path does not appear: the PDE is a
# statement about the pricing function on the whole state space."
# The mesh is therefore disjoint from the data manifold, and L_f gets its own
# resolution instead of being tied to observation density.
N_COLLOCATION = 4096
DELTA_MESH_RANGE = (-1.0, 1.5)   # covers delta_true across all 100 paths: [-0.98, 1.36]

_key_mesh = random.PRNGKey(0)
_k1, _k2, _k3 = random.split(_key_mesh, 3)
S_mesh = random.uniform(_k1, (N_COLLOCATION,),
                        minval=float(S_train.min()), maxval=float(S_train.max()))
delta_mesh = random.uniform(_k2, (N_COLLOCATION,),
                            minval=DELTA_MESH_RANGE[0], maxval=DELTA_MESH_RANGE[1])
tau_mesh = random.uniform(_k3, (N_COLLOCATION,), minval=0.0, maxval=float(taus.max()))
mesh = (S_mesh, delta_mesh, tau_mesh)

print("t_train        ", t_train.shape, f"[{float(t_train.min()):.2f}, {float(t_train.max()):.2f}]")
print("S_train        ", S_train.shape, f"[{float(S_train.min()):.2f}, {float(S_train.max()):.2f}]")
print("taus           ", taus.shape, f"[{float(taus.min()):.3f}, {float(taus.max()):.3f}]")
print("log_F_obs_train", log_F_obs_train.shape, "(dates x maturities)")
print("delta_true     ", delta_true.shape, "HELD OUT -- evaluation only")
print("collocation    ", S_mesh.shape, "points over (S, delta, tau), delta independent")


t_train         (1001,) [0.00, 10.00]
S_train         (1001,) [35.89, 165.12]
taus            (12,) [0.083, 1.000]
log_F_obs_train (1001, 12) (dates x maturities)
delta_true      (1001,) HELD OUT -- evaluation only
collocation     (4096,) points over (S, delta, tau), delta independent


### Loss terms — `error()`

Function adapted from Dupire usecase to use the gibson-schwartz loss functions detailed in `docs/mathematical_derivations-5.pdf`.

- `e_acc` = their $\mathcal{L}_t$ (eq. 32)
- `e_pde` = their $\mathcal{L}_f$ (eq. 33/35)
- `e_arb_cac` = their $\mathcal{L}_{h,\text{cac}}$, cash-and-carry upper bound (eq. 22)
- `e_arb_rcac` = their $\mathcal{L}_{h,\text{rcac}}$, reverse-cash-and-carry lower bound (eq. 25)
- `e_delta_floor` = their $\mathcal{L}_{h,\delta\text{-floor}}$, convenience-yield floor (eq. 28)
- $\mathcal{L}_b$ is **not** implemented

In [35]:
# CONSTANTS
STORAGE_COST_U = 0.0    # u >= 0, storage+insurance rate (S7.1). Paper gives no number.
DELTA_MAX = 1.5         # rcac convenience-yield wedge (S7.2, eq. 36). Paper gives no number.

# S7.3 eq. 39 gives delta_min = -0.3, justified as "the empirical lower edge of
# the convenience-yield range in the Monte Carlo simulated data" and claimed to
# return "zero on all admissible states" (eq. 40).
# CHECKED AGAINST mc_data.pkl AND IT DOES NOT HOLD: delta_true spans [-0.98, 1.36]
# across the 100 paths, 1st pct = -0.51, and 6.27% of true states lie BELOW -0.3
# (4.30% on path 0 alone). At -0.3 the floor therefore penalises the ground
# truth and biases delta_hat_phi upward in exactly the deep-contango regime
# S7.3 says the model is meant to capture.
# Paper value kept so the code matches the write-up; resolve before citing eq. 39.
DELTA_MIN = -0.3
# DELTA_MIN = -0.51   # 1st percentile: regularises without binding on plausible states


def error(fns, data, mesh):
    """Loss residuals, eq. 49.

    fns  : (F_hat_fn, delta_hat_fn, log_F_hat_fn) from make_fns
    data : (t_train, S_train, log_F_obs_train)  -- the DATA MANIFOLD
    mesh : (S_mesh, delta_mesh, tau_mesh)       -- the COLLOCATION MESH

    Returns (err, metrics), same contract as before.
    """
    F_hat_fn, delta_hat_fn, log_F_hat_fn = fns
    t_train, S_train, log_F_obs_train = data
    S_mesh, delta_mesh, tau_mesh = mesh

    # Latent path at observation times. Input is t alone (S6 remark 1).
    delta_hat = vmap(delta_hat_fn)(t_train)                          # (n,)

    # --- e_acc  (L_t, eq. 42-43) -------------------------------------------
    # Priced through the SURFACE NET at the inferred state
    # x_i = (S_i, delta_hat_phi(t_i), tau_i) -- not through the closed form.
    # delta_hat enters via dF_theta/ddelta, which is the sole channel by which
    # observations discipline the path (S9).
    log_F_hat = vmap(
        lambda S, d: vmap(lambda tau: log_F_hat_fn(S, d, tau))(taus)
    )(S_train, delta_hat)                                            # (n, K)
    e_acc = (log_F_hat - log_F_obs_train) ** 2                       # (n, K)

    # --- e_pde  (L_f, eq. 44-45) -------------------------------------------
    # On the MESH, with delta an independent coordinate. All partials by
    # autodiff on the free surface. The latent path does not appear.
    def pde_residual(S, delta, tau):
        F_tau   = grad(F_hat_fn, 2)(S, delta, tau)
        F_S     = grad(F_hat_fn, 0)(S, delta, tau)                   # delta held FIXED
        F_SS    = grad(grad(F_hat_fn, 0), 0)(S, delta, tau)
        F_d     = grad(F_hat_fn, 1)(S, delta, tau)
        F_dd    = grad(grad(F_hat_fn, 1), 1)(S, delta, tau)
        F_Sd    = grad(grad(F_hat_fn, 0), 1)(S, delta, tau)
        return (F_tau
                - (p_Q.r - delta) * S * F_S
                - p_Q.kappa * (p_Q.alpha_Q - delta) * F_d
                - 0.5 * p_Q.sigma1 ** 2 * S ** 2 * F_SS
                - p_Q.rho * p_Q.sigma1 * p_Q.sigma2 * S * F_Sd
                - 0.5 * p_Q.sigma2 ** 2 * F_dd)

    e_pde = vmap(pde_residual)(S_mesh, delta_mesh, tau_mesh) ** 2     # (N_f,)

    # --- Carry bounds, on the DATA MANIFOLD (S7 "Domain of enforcement") ----
    # Not on the mesh: the cac ceiling makes no reference to delta, and by
    # eq. 35 the true GS surface violates it wherever delta < ((r+u)tau - A)/B.
    # Enforcing it over the full mesh would contradict the PDE residual.
    F_hat = jnp.exp(log_F_hat)                                        # (n, K)

    # eq. 34: cash-and-carry ceiling. Note it does NOT reference delta -- a
    # model-free bound must not depend on the latent state being inferred.
    cac_bound = S_train[:, None] * jnp.exp((p_Q.r + STORAGE_COST_U) * taus[None, :])
    e_arb_cac = jnp.maximum(0.0, F_hat - cac_bound) ** 2              # (n, K)

    # eq. 37: reverse cash-and-carry floor, carrying the delta_max wedge. The
    # naive F >= S e^{r tau} is recovered only as delta_max -> 0; backwardation
    # is the normal signature of a positive convenience yield, not an arbitrage.
    rcac_bound = S_train[:, None] * jnp.exp((p_Q.r - DELTA_MAX) * taus[None, :])
    e_arb_rcac = jnp.maximum(0.0, rcac_bound - F_hat) ** 2            # (n, K)

    # eq. 38/40: floor on the latent path itself, no S or tau dependence.
    e_delta_floor = jnp.maximum(0.0, DELTA_MIN - delta_hat) ** 2      # (n,)

    err = {
        "e_acc": e_acc,
        "e_pde": e_pde,
        "e_arb_cac": e_arb_cac,
        "e_arb_rcac": e_arb_rcac,
        "e_delta_floor": e_delta_floor,
    }
    metrics = {k: jnp.mean(v) for k, v in err.items()}
    return err, metrics


In [36]:
def adj(loss, lw, m):
    return lw * jnp.mean(m * loss)


def make_loss_lb(components):
    def loss_fn(fns, data, mesh, l_ws, params_sa):
        err, metrics = error(fns, data, mesh)
        loss = {k: adj(err[k], l_ws[k], params_sa[k]) for k in components}
        return loss, metrics
    return loss_fn


def make_loss_fn(components):
    def loss_fn(fns, data, mesh, l_ws, params_sa):
        loss, metrics = loss_fn_lb[components](fns, data, mesh, l_ws, params_sa)
        return sum(loss.values()), metrics
    return loss_fn


# MLP:    data fit only -- ablation. The surface is unconstrained by physics, so
#         this asks whether fitting the panel alone already pins delta.
# PINN:   data fit + PDE residual on the mesh.
# ACPINN: PINN + the three inequality terms of S7.
# e_b is absent everywhere: the eq. 47 output transform enforces F(S,delta,0)=S
# identically, so L_b is dropped from objective (41) by construction.
loss_fn_lb = {
    "MLP":    make_loss_lb(["e_acc"]),
    "PINN":   make_loss_lb(["e_acc", "e_pde"]),
    "ACPINN": make_loss_lb(["e_acc", "e_pde", "e_arb_cac", "e_arb_rcac", "e_delta_floor"]),
}

loss_fn = {k: make_loss_fn(k) for k in loss_fn_lb}


### Calibration

The training loop itself — `TrainState`, the self-adaptive per-point weight optimizer (SGD ascent on `params_sa`), the gradient-norm loss-balancing ("Whack-a-mole"), and checkpointing — is ported essentially verbatim; none of it references SABR/Black-Scholes. What changed:
- `init_l_ws` / `init_params_sa`: component names and per-point weight shapes now match `error()`'s `e_acc` `(n, K)` / `e_pde` `(n,)` instead of the original's `x_train`/`x_mesh` shapes. **TODO**: add a `"ACPINN"` entry to both once you've added inequality terms.
- `s_0`, `r` are gone as explicit args — pricing params now live in the closed-over `p_Q`/`kappa_P`/`alpha_P` globals from the Input Data cell.
- `run_experiment` takes `data` directly instead of re-sampling it via `get_data(config.pts_num, ...)` each call, since our dataset is a fixed loaded panel, not something to redraw per experiment.

In [ ]:
def save_params(params, path_item: str) -> None:
    serialized_params = to_state_dict(params)
    with open(path_item, 'wb') as f:
        pickle.dump(serialized_params, f)

def load_params(params_initialized, path_item: str):
    with open(path_item, 'rb') as f:
        loaded_dict = pickle.load(f)
    return from_state_dict(loaded_dict, params_initialized)

def flatten_pytree(pytree):
    return ravel_pytree(pytree)[0]

init_l_ws = {
    "MLP": {'e_acc': 1.},
    "PINN": {'e_acc': 1., 'e_pde': 1.},
    "ACPINN": {'e_acc': 1., 'e_pde': 1., 'e_arb_cac': 1., 'e_arb_rcac': 1., 'e_delta_floor': 1.},
}


def init_params_sa(loss_str, data, mesh):
    """Pointwise self-adaptive weights m, eq. 48, m(0) = 1.

    Shapes now differ by DOMAIN, which is the substantive change: e_acc and the
    carry terms live on the data manifold (n, K); e_pde lives on the collocation
    mesh (N_f,); the delta floor lives on the path (n,).
    """
    t_train, S_train, log_F_obs_train = data
    n, k = log_F_obs_train.shape
    n_f = mesh[0].shape[0]
    ret = {
        "MLP": {'e_acc': jnp.ones((n, k))},
        "PINN": {'e_acc': jnp.ones((n, k)),
                 'e_pde': jnp.ones(n_f)},
        "ACPINN": {'e_acc': jnp.ones((n, k)),
                   'e_pde': jnp.ones(n_f),
                   'e_arb_cac': jnp.ones((n, k)),
                   'e_arb_rcac': jnp.ones((n, k)),
                   'e_delta_floor': jnp.ones(n)},
        }
    return ret[loss_str]


def calibration(config, data, mesh):
    ofunc = loss_fn[config.loss_str]

    l_ws = dict(init_l_ws[config.loss_str])
    params_sa = init_params_sa(config.loss_str, data, mesh)

    key = random.PRNGKey(config.seed)
    key, key_init = random.split(key, 2)

    # One TrainState over the joint pytree {"surface": theta, "path": phi} --
    # this is what "trained jointly" (S6) means operationally: a single
    # apply_gradients updates both nets from the one composite objective.
    surface_ann, path_ann, params_init = init_nets(config, key_init)
    state = train_state.TrainState.create(apply_fn=None, params=params_init, tx=tx)

    # self-adaptive
    opt_init_sa, opt_update_sa, get_params_sa = jax_optimizers.sgd(1.0)
    state_sa = opt_init_sa(params_sa)

    @jit
    def train_step(state, data, mesh, l_ws, state_sa):
        params_sa = get_params_sa(state_sa)
        def loss_fn_(params):
            fns = make_fns(surface_ann, path_ann, params)
            return ofunc(fns, data, mesh, l_ws, params_sa)

        (loss, metric), grads = jax.value_and_grad(loss_fn_, has_aux=True)(state.params)
        state = state.apply_gradients(grads=grads)
        return state, loss, metric, state_sa

    hist_loss = []
    momentum = config.loss_balancing_momentum
    start_time = time.time()

    @jit
    def train_step_sa(step, state, data, mesh, l_ws, state_sa):
        fns = make_fns(surface_ann, path_ann, state.params)
        def loss_fn_sa(params_sa):
            return ofunc(fns, data, mesh, l_ws, params_sa)[0]

        params_sa = get_params_sa(state_sa)
        value_sa, grads_sa = jax.value_and_grad(loss_fn_sa)(params_sa)
        for key_ in grads_sa.keys():
            grads_sa[key_] *= -1.          # ascent on m (eq. 50 saddle problem)
        state_sa = opt_update_sa(step, grads_sa, state_sa)
        return state_sa

    @jit
    def update_loss_weights(state, data, mesh, l_ws, state_sa):
        params_sa = get_params_sa(state_sa)
        def loss_fn_(params):
            fns = make_fns(surface_ann, path_ann, params)
            return loss_fn_lb[config.loss_str](fns, data, mesh, l_ws, params_sa)[0]
        grads = jacrev(loss_fn_)(state.params)

        # NOTE (S12.2, eq. 55-56): this takes the mean |grad| over the FULL
        # parameter vector psi = (theta, phi). The paper requires each category's
        # gradient scale to be measured on its own support: psi_f = theta,
        # psi_delta-floor = phi, psi_t = psi_cac = psi_rcac = (theta, phi).
        # Under two nets with dim phi << dim theta this dilutes g_delta-floor by
        # dim phi/dim psi and inflates lambda_delta-floor by the reciprocal --
        # the paper puts the distortion at one to two orders of magnitude.
        # Left as-is because the WamOL balancing rewrite is PARKED in
        # pinn_rewrite_todo.md; fix it there, not here.
        grad_norm_dict = {}
        for key_, value in grads.items():
            flattened_grad = flatten_pytree(value)
            grad_norm_dict[key_] = jnp.abs(flattened_grad).mean()

        sum_grad_norm = jnp.sum(jnp.stack(tree_leaves(grad_norm_dict)))
        w = tree_map(lambda x: jnp.where(x==0., 1., sum_grad_norm / x), grad_norm_dict)

        running_average = (
            lambda old_w, new_w: old_w * momentum + (1 - momentum) * new_w
        )
        weights = tree_map(running_average, l_ws, w)
        weights = lax.stop_gradient(weights)
        return weights

    # Training loop
    print(f"{config.loss_str} calibration------>")
    # config.balancing gates the WamOL whack-a-mole (lambda-balancing +
    # self-adaptive m). Only ACPINN carries the inequality terms it was built
    # for, so it is the only variant that ever ran the balancer. Setting it
    # False runs ACPINN with fixed lambda=1, m=1 -- the control that isolates
    # whether the balancer, not the constraints, drives the path collapse.
    use_balancing = config.get("balancing", True) and config.loss_str == "ACPINN"
    for epoch in range(config.num_epochs):

        # Whack-a-mole Learning
        if use_balancing and epoch % 100 == 0:
            l_ws = update_loss_weights(state, data, mesh, l_ws, state_sa)
        if use_balancing and (epoch+50) % 100 == 0:
            state_sa = train_step_sa(epoch, state, data, mesh, l_ws, state_sa)

        state, loss, metric, state_sa = train_step(state, data, mesh, l_ws, state_sa)

        if (epoch % 1000) == 0:
            print(f"Epoch {epoch}: loss = {loss:.6f}", end="\r")
            hist_loss.append((epoch, float(loss), metric))

    comp_time = time.time() - start_time
    print(f"------> completed in {comp_time:.2f} seconds")

    # Save checkpoint
    CKPT_DIR = os.path.abspath(f'../data/output/checkpoints/{run_tag(config)}')
    ckpt = {'params': state.params, 'ms': get_params_sa(state_sa), 'ls': l_ws}
    orbax_checkpointer = orbax.checkpoint.PyTreeCheckpointer()
    save_args = orbax_utils.save_args_from_target(ckpt)
    orbax_checkpointer.save(CKPT_DIR, ckpt, force=True, save_args=save_args)

    # Return the bound callables, not a single `fn` -- downstream code needs the
    # surface and the path separately.
    return make_fns(surface_ann, path_ann, state.params), hist_loss


def run_tag(config):
    """File-safe name distinguishing ACPINN with balancing on vs off."""
    if config.loss_str == "ACPINN" and not config.get("balancing", True):
        return "ACPINN_nobal"
    return config.loss_str


def run_experiment(config, data, mesh):
    fns, hist_loss = calibration(config, data, mesh)

    results = {'config': config, 'history': hist_loss}
    with open(f'../data/output/results/results_{run_tag(config)}.pkl', 'wb') as f:
        pickle.dump(results, f)

    return fns


### Run

`ann_in_dim=2` (`t, S`), `ann_out_dim=1` (`delta_hat`) — matches eq. (32)'s `delta_hat(t_i, S_i)`. `pts_num` is dropped: the original resampled a fixed-size training batch from its synthetic distribution each run; here the whole path is used as one batch, so there's no equivalent knob (add batching/subsampling here yourself if `n` gets large enough to matter).

`num_epochs=3000` below is a smoke-test setting to prove the wiring runs end to end — not a converged model, and ACPINN in particular (5 loss terms, gradient-norm rebalancing every 100 epochs) will need more.

In [ ]:
# Two nets now, so there is no single `ann_in_dim`: the surface net is 3-D
# (S, delta, tau) and the path net is 1-D (t). Surface is the wider of the two --
# it carries the whole pricing function; the path is a scalar map of t.
config_MLP = ml_collections.ConfigDict({
    "ann_str": "MLP",
    "loss_str": "MLP",
    "num_epochs": 3000,
    "surface_hidden_dim": (64, 64, 64, 64),
    "path_hidden_dim": (32, 32, 32),
    "ann_activation_str": "tanh",
    "self_adaptive_lr": 1.0,
    "loss_balancing_momentum": 0.5,
    "seed": 42,
    "ann_reparam": False,
    "balancing": True,     # WamOL whack-a-mole on by default (ACPINN only)
})

config_PINN = ml_collections.ConfigDict(config_MLP.to_dict())
config_PINN.loss_str = "PINN"

config_ACPINN = ml_collections.ConfigDict(config_PINN.to_dict())
config_ACPINN.loss_str = "ACPINN"

# CONTROL: ACPINN with the WamOL balancer OFF. Same loss terms, same seed,
# fixed lambda=1 / m=1. If this recovers the path like PINN does while the
# balanced ACPINN collapses, the balancer -- not the constraints -- is the cause.
config_ACPINN_nobal = ml_collections.ConfigDict(config_ACPINN.to_dict())
config_ACPINN_nobal.balancing = False

fns_MLP = run_experiment(config_MLP, data, mesh)
fns_PINN = run_experiment(config_PINN, data, mesh)
fns_ACPINN = run_experiment(config_ACPINN, data, mesh)
fns_ACPINN_nobal = run_experiment(config_ACPINN_nobal, data, mesh)


In [ ]:
# compute summary metrics for each model
for fns, config in zip([fns_MLP, fns_PINN, fns_ACPINN, fns_ACPINN_nobal],
                       [config_MLP, config_PINN, config_ACPINN, config_ACPINN_nobal]):
    err, metrics = error(fns, data, mesh)
    print(f"Metrics for {run_tag(config)}:")
    for k, v in metrics.items():
        print(f"  {k}: {v:.6e}")


### Visualization

`plot_training_history` is ported verbatim — it's loss-component-agnostic (reads whatever metric keys are in the pickled history), so it works unmodified for MLP/PINN/ACPINN alike. Dropped entirely: `plot_volatility_surface`, `plot_arbitrage_heatmaps`, `compare_with_sabr` — all keyed on a `(K,T)` grid that doesn't exist here.

`plot_delta_recovery` is new — the GS analogue of `compare_with_sabr`'s "learned vs. true" comparison, now evaluating `model(t, S)` since the network takes both. `delta_true` is used here and only here, exactly as CLAUDE.md requires ("the true delta_t path is sacred... never an input to the inversion") — it's a post-hoc scoring plot, not something the network ever sees.

In [ ]:
def plot_training_history(history_path, save_path=None):
    """Plot training loss history"""
    with open(history_path, 'rb') as f:
        results = pickle.load(f)
    history = results['history']

    epochs, losses, metrics = zip(*history)

    fig, ax = plt.subplots(1, 1, figsize=(6, 4))
    ax.plot(epochs, losses, 'b-', label='Total Loss')

    for key in metrics[0].keys():
        metric_values = [m[key] for m in metrics]
        ax.plot(epochs, metric_values, '--', label=key)

    ax.set_xlabel('Epoch')
    ax.set_ylabel('Loss')
    ax.set_yscale('log')
    ax.grid(True)
    ax.legend()
    plt.title(f"Training History ({results['config'].loss_str})")

    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.show()
    plt.close()


def plot_delta_recovery(fns, t, delta_true, save_path=None):
    """Recovered delta_hat_phi(t) vs. the true (eval-only) delta_t path.

    The path net takes t alone now (S6 remark 1), so S is no longer an argument.
    """
    _, delta_hat_fn, _ = fns
    delta_hat = vmap(delta_hat_fn)(t)

    rmse = float(jnp.sqrt(jnp.mean((delta_hat - delta_true) ** 2)))

    fig, ax = plt.subplots(1, 1, figsize=(8, 4))
    ax.plot(t, delta_true, color='black', lw=1.2, label=r'$\delta_t$ (true)')
    ax.plot(t, delta_hat, color='tab:red', lw=1.2, ls='--', label=r'$\hat{\delta}_t$ (recovered)')
    ax.axhline(DELTA_MIN, color='tab:blue', lw=0.8, ls=':', label=r'$\delta_{\min}$ floor')
    ax.set_xlabel('t (years)')
    ax.set_ylabel('Convenience yield')
    ax.legend()
    ax.set_title(f'Recovered vs. true convenience yield (RMSE {rmse:.4f})')

    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.show()
    plt.close()


os.makedirs('figures', exist_ok=True)

plot_training_history('../data/output/results/results_PINN.pkl', '../figures/pinn/training_history_PINN.png')
plot_delta_recovery(fns_PINN, t_train, delta_true, '../figures/pinn/delta_recovery_PINN.png')

plot_training_history('../data/output/results/results_ACPINN.pkl', '../figures/pinn/training_history_ACPINN.png')
plot_delta_recovery(fns_ACPINN, t_train, delta_true, '../figures/pinn/delta_recovery_ACPINN.png')

# Control: ACPINN, balancer off.
plot_training_history('../data/output/results/results_ACPINN_nobal.pkl', '../figures/pinn/training_history_ACPINN_nobal.png')
plot_delta_recovery(fns_ACPINN_nobal, t_train, delta_true, '../figures/pinn/delta_recovery_ACPINN_nobal.png')
